# Bedding × Dinomaly — training tutorial

Walks through how to train the 6-channel ALL6 pipeline from scratch on the bedding dataset. For brevity this notebook runs a 1-epoch smoke; the production pilot is the 20-epoch orchestrator in `examples/bedding_dinomaly/run_pilot_all6_highres_30ep.sh`.

Covers:
1. Bedding dataset overview (6 channels, 252 frames, 23 anomaly classes).
2. Building the pipeline programmatically — `FixedHyperspectralSelector` → `MinMaxNormalizer` → `DinomalyDetector(input_channels=6)`.
3. Running a 1-epoch smoke via the production training script (`train_bedding_all6.py`).
4. Inspecting the saved `pipeline.yaml` + `.pt`.
5. (Optional) Swapping square squash for aspect-preserving training with a 1-line change.

## 1 · Dataset overview

- **Format**: cu3s session files, one per frame.
- **Channels**: 6 bands at 450 / 550 / 625 nm (VIS) and 1050 / 1200 / 1450 nm (SWIR). Same camera assembly as the cuvis.ai.examples bedding setup.
- **Spatial resolution**: 1800 × 4300 (aspect 2.39:1).
- **Scene**: a sawdust-filled tray with foreign objects — PLA cubes (multiple sizes + colours), leaves, plastic foil, PET, POMC, water, alcohol.
- **Splits**: 193 train + 59 val (no test). The train split contains only normal frames (unsupervised anomaly detection setup). The val split has both normal `_ok_ok_` frames and anomalous frames with various foreign-object configurations.
- **Anomaly classes**: 23 (see `comparisons/per_class_auroc_matrix.md` for the full list).

In [ ]:
%matplotlib inline
import subprocess, sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))
from utils import resolve_default_config, BEDDING_ALL6_NM, BEDDING_ALL6_LABELS

config = resolve_default_config()
print('Plugins manifest:', config['plugins_yaml'])
print('Splits CSV:      ', config['splits_csv'])
print('Target λ order:  ', BEDDING_ALL6_NM)
print('Channel labels:  ', BEDDING_ALL6_LABELS)

## 2 · Pipeline shape

The ALL6 pipeline is wired together inside `train_bedding_all6.py`. The graph is:

```
LentilsAnomalyDataNode               # reads NPZ cubes from disk
 → MinMaxNormalizer                  # cube → [0, 1]
 → FixedHyperspectralSelector(       # pick + reorder bands
       target_wavelengths=(625, 550, 450, 1450, 1200, 1050),
       normalize_output=False)
 → DinomalyDetector(                 # 6-ch DINOv2 + bottleneck + decoder
       input_channels=6,             # patch-embed inflated 3→6
       image_size=672,               # square squash (set to (H, W) for aspect-preserving)
       use_center_crop=False)
 → decider                           # binary anomaly decision
```

Two implementation notes from the bedding × Dinomaly work:

- **`input_channels=6` is enabled by patch-embed inflation**: the DINOv2 encoder ships with a `Conv2d(3, 768, 14, stride=14)` patch projector. To accept 6 channels we duplicate the conv weights along the input axis and halve them so a 6-channel input that satisfies `x[3:6] ~ x[0:3]` produces identical activations to the 3-channel model at init. The inflated conv is the only encoder weight unfrozen during training; the rest of DINOv2 stays frozen. See `cuvis_ai_dinomaly/node/_patch_embed_inflation.py`.
- **`image_size` accepts `int | (h, w)` tuple**: int is square squash (backward-compatible default); tuple is aspect-preserving. For non-square inputs we monkey-patch anomalib's `DinomalyModel.get_encoder_decoder_outputs` to use input-derived `(H_p, W_p)` instead of `sqrt(N_tokens)` (which assumes square). See `cuvis_ai_dinomaly/node/_rectangular_input_patch.py`.

## 3 · 1-epoch smoke training

Invokes `train_bedding_all6.py` as a subprocess with `--max-epochs 1` and a tutorial-specific output directory. The full 20-epoch production pilot takes ~3h on RTX A4000 (16 GB) at 672×672 fp32; the 1-epoch smoke takes ~3–5 minutes.

The script handles the full lifecycle: data loading, pipeline construction, Lightning training loop, validation per epoch, AUROC metrics logging via the `AnomalyAUROCMetrics` callback, and finally serialisation of the pipeline to YAML + .pt.

In [ ]:
TUTORIAL_OUT = Path('outputs/smoke_train_run')
TUTORIAL_OUT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    '/home/dev/anish/cuvis-ai-cookbook/examples/bedding_dinomaly/train_bedding_all6.py',
    '--splits-csv', str(config['splits_csv']),
    '--output-dir', str(TUTORIAL_OUT.resolve()),
    '--max-epochs', '1',
    '--precision', '32-true',
    '--num-workers', '0',
    '--image-size', '448',
]
print('Running:', ' '.join(cmd))
print('(this takes ~3-5 minutes; output streamed to outputs/smoke_train_run/train.log)')

with open(TUTORIAL_OUT / 'train.log', 'w') as log:
    result = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, check=False)
print(f'\nReturn code: {result.returncode}')
print(f'Log size: {(TUTORIAL_OUT / "train.log").stat().st_size:,} bytes')

In [ ]:
# Inspect what landed in the output directory.
for p in sorted(TUTORIAL_OUT.rglob('*'))[:20]:
    if p.is_file():
        rel = p.relative_to(TUTORIAL_OUT)
        print(f'  {p.stat().st_size:>10,} B  {rel}')

In [ ]:
# Tail the train log to see the final epoch's val metrics.
log_path = TUTORIAL_OUT / 'train.log'
if log_path.is_file():
    lines = log_path.read_text().splitlines()
    # Pull metric lines near the end of the run.
    keep = [l for l in lines if any(k in l for k in ('auroc_pixel', 'auroc_image',
                                                       'metrics_anomaly', 'val_loss',
                                                       'Val epoch AUROC'))]
    for l in keep[-20:]:
        print(l)

## 4 · Saved artefacts

After training, `train_bedding_all6.py` saves a serialised pipeline at `<output-dir>/trained_models/dinomaly_bedding_all6.yaml` plus the corresponding `.pt` weights. These two files are everything you need to load the model back later — see `bedding_all6_inference_tutorial.ipynb`.

In [ ]:
yaml_path = TUTORIAL_OUT / 'trained_models' / 'dinomaly_bedding_all6.yaml'
if yaml_path.is_file():
    text = yaml_path.read_text()
    # Show only the first ~40 lines so the notebook output stays readable.
    print('\n'.join(text.splitlines()[:40]))
else:
    print(f'No pipeline YAML at {yaml_path} (training may not have completed cleanly — check train.log)')

## 5 · (Optional) Aspect-preserving training

To swap the default square squash for aspect-preserving training, pass two ints to `--image-size`:

```bash
python train_bedding_all6.py \
    --splits-csv /mnt/data/bedding_dataset_npz/bedding_splits_npz.csv \
    --output-dir /mnt/data/cuvis_ai_outputs/dinomaly_bedding_all6_aspect_434x1036_20ep \
    --max-epochs 20 \
    --precision 32-true \
    --num-workers 0 \
    --image-size 434 1036   # 31×74 patches @ patch=14, aspect 2.39 — matches the bedding cube
```

Both ints must be multiples of the encoder's patch size (14 for `dinov2reg_vit_base_14`). The `_rectangular_input_patch` monkey-patch is only applied when the two dimensions differ, so the int form is exactly the prior square behaviour (backward compatible).

See `examples/bedding_dinomaly/run_pilot_all6_aspect_dual_20ep.sh` for a production orchestrator running two aspect-preserving variants sequentially.